In [13]:
from datetime import UTC, datetime, timedelta
from io import StringIO

import lxml
import pandas as pd
import requests
from lxml import html


In [14]:
def format_date(date_datetime):
    """Convert a datetime object to a string representing epoch time.

    Args:
        date_datetime (datetime): The datetime object to convert.

    Returns:
        str: The epoch time as a string.
    """
    return str(int(date_datetime.timestamp()))


def subdomain_div_history(symbol, start, end):
    """Generate the subdomain URL for the dividend history page on Yahoo Finance. Start and end parameters are used to filter time period of the dividends.

    Args:
        symbol (str): The stock symbol.
        start (str): The start date in epoch time as a string.
        end (str): The end date in epoch time as a string.

    Returns:
        str: The subdomain URL for the dividend history page.
    """
    format_url = "{0}/history?period1={1}&period2={2}"
    # URL for filters
    tail_url = "&interval=div%7Csplit&filter=div&frequency=1d"
    subdomain = format_url.format(symbol, start, end) + tail_url
    return subdomain


def subdomain_statistics(symbol):
    """Generate the subdomain URL for the key statistics page on Yahoo Finance.

    Args:
        symbol (str): The stock symbol.

    Returns:
        str: The subdomain URL for the key statistics page.
    """
    format_url = "{0}/key-statistics"
    return format_url.format(symbol)

def subdomain_financials(symbol):
    """Generate the subdomain URL for the financials page on Yahoo Finance.

    Args:
        symbol (str): The stock symbol.

    Returns:
        str: The subdomain URL for the financials page.
    """
    format_url = "{0}/financials"
    return format_url.format(symbol)


def header(subdomain):
    """Generate the headers for a request to the given subdomain on Yahoo Finance.

    Args:
        subdomain (str): The subdomain URL for the request.

    Returns:
        dict: The headers to use for the request.
    """
    hdrs = {
        "authority": "finance.yahoo.com",
        "method": "GET",
        "path": subdomain,
        "scheme": "https",
        "accept": "text/html,application/xhtml+xml",
        "accept-encoding": "gzip, deflate, br",
        "accept-language": "en-US,en;q=0.9",
        "cache-control": "no-cache",
        "dnt": "1",
        "pragma": "no-cache",
        "sec-fetch-mode": "navigate",
        "sec-fetch-site": "same-origin",
        "sec-fetch-user": "?1",
        "upgrade-insecure-requests": "1",
        "user-agent": "Mozilla/5.0",
    }
    return hdrs


def scrape_page(url, header):
    """Scrape the HTML content of the given URL and extract the first table as a DataFrame.

    Args:
        url (str): The URL to scrape.
        header (dict): The headers to use for the request.

    Returns:
        list: A list containing a DataFrame of the first table found on the page.
    """
    page = requests.get(url, headers=header)
    element_html = html.fromstring(page.content)
    table = element_html.xpath("//table")
    table_tree = lxml.etree.tostring(table[0], method="xml", encoding="unicode")
    panda = pd.read_html(StringIO(table_tree))
    return panda

def scrape_page_financials(url, header):
    """Scrape the HTML content of the given URL and extract values in div elements for the financials page.

    Args:
        url (str): The URL to scrape.
        header (dict): The headers to use for the request.

    Returns:
        pd.DataFrame: A DataFrame of the values extracted from the div elements on the page.
    """
    page = requests.get(url, headers=header)

    # This element is not a table. Had to do some extra parsing to extract from div structure.
    element_html = html.fromstring(page.content)
    table = element_html.xpath("//div[@class='tableBody yf-bw8utn']")
    rows = table[0].xpath(
        ".//div[contains(concat(' ', normalize-space(@class), ' '), ' row ')]"
    )

    # Parse child div elements of div container
    fins = []
    for row in rows:
        cells = row.xpath("./div")  # assumes each cell is a direct child of the row
        values = [" ".join(cell.itertext()).strip() for cell in cells]
        if values:
            fins.append(values)

    panda = pd.DataFrame(fins)
    return panda


def clean_dividends(dividends):
    """Clean the dividend history table by removing unnecessary columns and formatting the data.

    Args:
        dividends (list[pd.DataFrame]): The raw dividend history table as a list of DataFrames.

    Returns:
        pd.DataFrame: The cleaned and formatted dividend history table.
    """
    dividends = pd.DataFrame(dividends)
    dividends = dividends.rename(columns={dividends.columns[1]: "Dividends"})
    dividends = dividends.drop(columns=dividends.columns[[2, 3, 4, 5, 6]])
    dividends["Dividends"] = dividends["Dividends"].str.replace("Dividend", "")
    dividends["Date"] = pd.to_datetime(dividends["Date"]).dt.strftime("%m-%d-%Y")
    dividends["Dividends"] = dividends["Dividends"].astype(float)
    dividends["Dividends"] = dividends["Dividends"].apply(lambda x: f"${x:,.2f}")
    return dividends


def clean_statistics(statistics: list[pd.DataFrame]):
    """Clean the key statistics table by removing unnecessary columns and formatting the data.

    Args:
        statistics (list[pd.DataFrame]): The raw key statistics table as a list of DataFrames.

    Returns:
        pd.DataFrame: The cleaned and formatted key statistics table.
    """
    statistics = pd.DataFrame(statistics)
    statistics = statistics.rename(columns={statistics.columns[0]: ""})
    statistics = statistics.reset_index(drop=True)
    return statistics

def clean_financials(financials: pd.DataFrame):
    """Clean the financials table by removing unnecessary columns and formatting the data.

    Args:
        financials (pd.DataFrame): The raw financials table as a DataFrame.

    Returns:
        pd.DataFrame: The cleaned and formatted financials table.
    """

    financials = financials.rename(
        columns={
            financials.columns[0]: "Breakdown",
            financials.columns[1]: "TTM",
            financials.columns[2]: f"12/31/{datetime.now(UTC).year - 1}",
            financials.columns[3]: f"12/31/{datetime.now(UTC).year - 2}",
            financials.columns[4]: f"12/31/{datetime.now(UTC).year - 3}",
            financials.columns[5]: f"12/31/{datetime.now(UTC).year - 4}",
            }
    )

    financials["TTM"] = financials["TTM"].replace(",","",regex=True).astype(float)
    financials["TTM"] = financials["TTM"].map("${:,.0f}".format)
    return financials

In [15]:
symbol = "VZ"
# create datetime objects
start = datetime.today() - timedelta(days=9125)  # noqa: DTZ002
end = datetime.today()  # noqa: DTZ002
# properly format the date to epoch time
start = format_date(start)
end = format_date(end)
# format the subdomain
sub_div = subdomain_div_history(symbol, start, end)
# customize the request header
hdrs_div = header(sub_div)

# concatenate the subdomain with the base URL
base_url = "https://finance.yahoo.com/quote/"
url = base_url + sub_div
# scrape the dividend history table from Yahoo Finance
dividends = scrape_page(url, hdrs_div)

# clean the dividend history table
clean_div = clean_dividends(dividends[0])

# scrpae key statistics subdomain
sub_stats = subdomain_statistics(symbol)
hdrs_stats = header(sub_stats)
url_stats = base_url + sub_stats
key_statistics = scrape_page(url_stats, hdrs_stats)

# format key statistics table
stats = clean_statistics(key_statistics[0])

# scrape financials subdomain
sub_financials = subdomain_financials(symbol)
hdrs_financials = header(sub_financials)
url_financials = base_url + sub_financials
financials = scrape_page_financials(url_financials, hdrs_financials)

# format financials table
financials = clean_financials(financials)

In [ ]:
# if files already exist, overwrite them
clean_div.to_csv("Dividend_Data.csv", index=False, mode="w")
stats.to_csv("Key_Statistics_Data.csv", index=False, mode="w")
financials.to_csv("Financials_Data.csv", index=False, mode="w")

### Exploratory Data Analysis

Something to score each data point in key_statistics and financials_data files. May trend data, but only 4 data points for each category.

Average dividend payout history. Based on history, trend and predict future dividend payout.